# Week 21: High-Performance ML Inference, Compilers & Serving
## Notebook 06: Medusa: Multi-Head Speculative Decoding from Scratch

### 1. Conceptual Intuition & Real-World Use Case
Speculative Decoding accelerates LLM generation:
- Standard speculative decoding uses a separate small "draft model" to propose candidate tokens, and the large model verifies them in one forward pass.
- **Problem:** Serving two separate models (draft + target) complicates deployment, increases VRAM usage, and creates latency bottlenecks if the draft model is too large.

In 2024, Cai et al. (Together AI) introduced **Medusa**:
- Eliminates the separate draft model entirely!
- Adds multiple lightweight **Medusa Heads** (single-layer feedforward heads) to the top of the frozen base model!
- Head 1 predicts token $t+1$, Head 2 predicts $t+2$, Head 3 predicts $t+3$ simultaneously in a single forward pass.
- Verified in parallel using **Tree-based Attention Verification**, achieving **2x to 3x real-world speedups**!

---

### 2. Mathematical Foundations: Multi-Token Heads
Given the target model's final hidden state $h_t$ at step $t$:

$$\hat{y}_{t+1} = \text{Softmax}(W_{\text{base}} h_t)$$
$$\hat{y}_{t+2} = \text{Softmax}(W_{\text{medusa}, 1} h_t)$$
$$\hat{y}_{t+3} = \text{Softmax}(W_{\text{medusa}, 2} h_t)$$

In the next forward pass, candidate token sequences are passed as a candidate tree. The base model evaluates all candidate tokens in a single parallel verification pass, accepting the prefix that matches greedy base predictions!

In [ ]:
import numpy as np

# Step 1: Simulate Target Base Model and 2 Medusa Draft Heads
vocab = ["The", "capital", "of", "France", "is", "Paris", "beautiful", "city"]
token_to_id = {w: i for i, w in enumerate(vocab)}
id_to_token = {i: w for i, w in enumerate(vocab)}

# Suppose current input is: "The capital of France is"
# Target base model predicts the next token, while Medusa heads predict t+1 and t+2
draft_head_predictions = {
    "Head_1 (t+1)": "Paris",
    "Head_2 (t+2)": "beautiful"
}

print("Base Input: 'The capital of France is'")
print(f"Medusa Draft Proposals -> Step t+1: '{draft_head_predictions['Head_1 (t+1)']}', Step t+2: '{draft_head_predictions['Head_2 (t+2)']}'")

# Step 2: Single Parallel Verification Forward Pass
# The base model evaluates the candidate sequence ['Paris', 'beautiful'] simultaneously in ONE step!
def verify_speculative_tokens(candidate_tokens, ground_truth_base_outputs):
    accepted_tokens = []
    for cand, truth in zip(candidate_tokens, ground_truth_base_outputs):
        if cand == truth:
            accepted_tokens.append(cand)
        else:
            # Rejection: discard all subsequent draft tokens and take ground truth
            accepted_tokens.append(truth)
            break
    return accepted_tokens

# Case A: Both draft tokens are correct ("Paris", "beautiful")
truth_case_A = ["Paris", "beautiful", "city"]
accepted_A = verify_speculative_tokens(["Paris", "beautiful"], truth_case_A)

print("\n--- Verification Scenario A (Full Acceptance) ---")
print(f"Accepted Tokens: {accepted_A}")
print(f"Tokens Generated in 1 Forward Pass: {len(accepted_A)} (Speedup: 3.0x!)")

# Case B: Head 2 made an incorrect guess (Truth was "a", not "beautiful")
truth_case_B = ["Paris", "a", "city"]
accepted_B = verify_speculative_tokens(["Paris", "beautiful"], truth_case_B)

print("\n--- Verification Scenario B (Partial Acceptance) ---")
print(f"Accepted Tokens: {accepted_B}")
print(f"Tokens Generated in 1 Forward Pass: {len(accepted_B)} (Speedup: 2.0x!)")

### 3. Step-by-Step Code Walkthrough
1. **Medusa Parallel Heads:** Multiple tokens are drafted simultaneously from a single base model forward pass.
2. **Parallel Tree Verification:** The base model verifies the drafted tokens in a single forward pass.
3. **Guaranteed Mathematical Equivalence:** Only drafted tokens that match the exact greedy output of the base model are accepted; any divergence triggers immediate fallback, preserving 100% output distribution fidelity.

---

### 4. Industry Applications & Trade-offs
- **High-Throughput Serving:** Medusa heads are trained in a few hours on top of frozen models without modifying base weights.
- **Trade-off:** Training auxiliary heads requires a high-quality dataset of domain text completions.

---

### 5. Top Interview Questions & Answers
**Q1: Why does Medusa achieve speedup even though extra heads add compute?**
> **Answer:** Autoregressive LLM inference is memory-bandwidth bound, not compute-bound: reading 70B parameters from VRAM takes 95% of step latency, while ALU compute takes 5%. Medusa uses the idle GPU compute capacity to evaluate multiple candidate tokens during that single VRAM read pass, producing 2-3 tokens per memory transfer.

**Q2: Does Speculative Decoding alter the model's output quality or perplexity?**
> **Answer:** No. Rejection sampling mathematically guarantees that the verified output sequence follows the exact same probability distribution as the target model evaluated autoregressively. Output quality and perplexity are completely identical.